# <font color="#0b486b">  FIT5215: Deep Learning (2025) - Assignment 2 (Section II)</font>
***
*CE/Lecturer (Clayton):*  **Dr Trung Le** | trunglm@monash.edu <br/>
*Lecturer (Clayton):* **A/Prof Zongyuan Ge** | zongyuan.ge@monash.edu <br/>
*Lecturer (Malaysia):*  **Dr Arghya Pal** | arghya.pal@monash.edu <br/>
 <br/>
*Head Tutor 3181:*  **Ms Ruda Nie H** |  \[RudaNie.H@monash.edu \] <br/>
*Head Tutor 5215:*  **Ms Leila Mahmoodi** |  \[leila.mahmoodi@monash.edu \]

<br/> <br/>
Faculty of Information Technology, Monash University, Australia
***

# <font color="0b486b">Assignment 2 – Deep Learning for Sequential Data</font>
### Due: <font color="red">11:55pm Sunday, 26 October 2025</font> (FIT5215)

#### <font color="red">Important note:</font> This is an **individual** assignment. It contributes **20%** to your final mark. Read the assignment instructions carefully.

## <font color="#0b486b">Assignment 2's Organization</font>
This assignment 2 has two (2) sections:
- Section 1: Fundamentals of RNNs (10 marks).
- Section 2: Deep Learning for Sequential Data (90 marks). This section is further divided into 4 parts.

The assignment 2 is organized in three (3) notebooks:
- Notebook 1 ([link](https://colab.research.google.com/drive/1hTS6bWz6gU-lvi1qkIhrtQnJ3tzxRPFK?usp=sharing)) [Total: 30 marks] includes Section 1 as well as Part 1 and Part 2 of Section 2.
- Notebook 2 (this notebook) [Total: 40 marks] includes Part 3 of Section 2.
- Notebook 3 ([link](https://colab.research.google.com/drive/1JfMZeCkkvjZ5LvKNV-UnR10pl-RogMgF?usp=sharing)) [Total: 30 marks] includes Part 4 of Section 2.


## <font color="#0b486b">What to submit</font>

This assignment is to be completed individually and submitted to Moodle unit site. **By the due date, you are required to submit one  <font color="red; font-weight:bold">single zip file, named xxx_assignment02_solution.zip</font> where `xxx` is your student ID, to the corresponding Assignment (Dropbox) in Moodle**. You can use Google Colab to do Assignment 2 but you need to save it to an `*.ipynb` file to submit to the unit Moodle.

**More importantly, if you use Google Colab to do this assignment, you need to first make a copy of this notebook on your Google drive**.

***For example, if your student ID is <font color="red; font-weight:bold">12356</font>, then gather all of your assignment solutions to a folder, create a zip file named <font color="red; font-weight:bold">123456_assignment02_solution.zip</font> and submit this file.***

Within this zip folder, you **must** submit the following files <u>for each part</u>:
1.	**`FIT5215_DeepLearning_Assignment2_Official[Main].ipynb`**:  this is your Python notebook solution source file.
1.	**`FIT5215_DeepLearning_Assignment2_Official[Main].html`**: this is the output of your Python notebook solution *exported* in HTML format.
1. **`FIT5215_DeepLearning_Assignment2_Official[RNNs].ipynb`**
1. **`FIT5215_DeepLearning_Assignment2_Official[RNNs].html`**
1. **`FIT5215_DeepLearning_Assignment2_Official[Transformers].ipynb`**
1. **`FIT5215_DeepLearning_Assignment2_Official[Transformers].html`**
1.	Any **extra files or folder** needed to complete your assignment (e.g., images used in your answers).



## Section 2: Deep Learning for Sequential Data

### <font color="#0b486b">Set random seeds</font>

We start with importing PyTorch and NumPy and setting random seeds for PyTorch and NumPy. You can use any seeds you prefer.

In [ ]:
import os
import torch
import random
import requests
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer
import os
from six.moves.urllib.request import urlretrieve
from sklearn import preprocessing
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [ ]:
def seed_all(seed=1029):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_all(seed=1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## <font color="#0b486b">Download and preprocess the data</font>

<div style="text-align: right"><font color="red; font-weight:bold"><span></div>

The dataset we use for this assignment is a question classification dataset for which the training set consists of $5,500$ questions belonging to 6 coarse question categories including:
- abbreviation (ABBR),
- entity (ENTY),
- description (DESC),
- human (HUM),
- location (LOC) and
- numeric (NUM).

In this assignment, we will utilize a subset of this dataset, containing $2,000$ questions for training and validation. We will use 80% of those 2000 questions for trainning and the rest for validation.


Preprocessing data is a crucial initial step in any machine learning or deep learning project. The *TextDataManager* class simplifies the process by providing functionalities to download and preprocess data specifically designed for the subsequent questions in this assignment. It is highly recommended to gain a comprehensive understanding of the class's functionality by **carefully reading** the content provided in the *TextDataManager* class before proceeding to answer the questions.

In [ ]:
class DataManager:
    """
    This class manages and preprocesses a simple text dataset for a sentence classification task.

    Attributes:
        verbose (bool): Controls verbosity for printing information during data processing.
        max_sentence_len (int): The maximum length of a sentence in the dataset.
        str_questions (list): A list to store the string representations of the questions in the dataset.
        str_labels (list): A list to store the string representations of the labels in the dataset.
        numeral_labels (list): A list to store the numerical representations of the labels in the dataset.
        numeral_data (list): A list to store the numerical representations of the questions in the dataset.
        random_state (int): Seed value for random number generation to ensure reproducibility.
            Set this value to a specific integer to reproduce the same random sequence every time. Defaults to 6789.
        random (np.random.RandomState): Random number generator object initialized with the given random_state.
            It is used for various random operations in the class.

    Methods:
        maybe_download(dir_name, file_name, url, verbose=True):
            Downloads a file from a given URL if it does not exist in the specified directory.
            The directory and file are created if they do not exist.

        read_data(dir_name, file_names):
            Reads data from files in a directory, preprocesses it, and computes the maximum sentence length.
            Each file is expected to contain rows in the format "<label>:<question>".
            The labels and questions are stored as string representations.

        manipulate_data():
            Performs data manipulation by tokenizing, numericalizing, and padding the text data.
            The questions are tokenized and converted into numerical sequences using a tokenizer.
            The sequences are padded or truncated to the maximum sequence length.

        train_valid_test_split(train_ratio=0.9):
            Splits the data into training, validation, and test sets based on a given ratio.
            The data is randomly shuffled, and the specified ratio is used to determine the size of the training set.
            The string questions, numerical data, and numerical labels are split accordingly.
            TensorFlow `Dataset` objects are created for the training and validation sets.


    """

    def __init__(self, verbose=True, random_state=6789):
        self.verbose = verbose
        self.max_sentence_len = 0
        self.str_questions = list()
        self.str_labels = list()
        self.numeral_labels = list()
        self.maxlen = None
        self.numeral_data = list()
        self.random_state = random_state
        self.random = np.random.RandomState(random_state)

    @staticmethod
    def maybe_download(dir_name, file_name, url, verbose=True):
        if not os.path.exists(dir_name):
            os.mkdir(dir_name)
        if not os.path.exists(os.path.join(dir_name, file_name)):
            urlretrieve(url + file_name, os.path.join(dir_name, file_name))
        if verbose:
            print("Downloaded successfully {}".format(file_name))

    def read_data(self, dir_name, file_names):
        self.str_questions = list()
        self.str_labels = list()
        for file_name in file_names:
            file_path= os.path.join(dir_name, file_name)
            with open(file_path, "r", encoding="latin-1") as f:
                for row in f:
                    row_str = row.split(":")
                    label, question = row_str[0], row_str[1]
                    question = question.lower()
                    self.str_labels.append(label)
                    self.str_questions.append(question[0:-1])
                    if self.max_sentence_len < len(self.str_questions[-1]):
                        self.max_sentence_len = len(self.str_questions[-1])

        # turns labels into numbers
        le = preprocessing.LabelEncoder()
        le.fit(self.str_labels)
        self.numeral_labels = np.array(le.transform(self.str_labels))
        self.str_classes = le.classes_
        self.num_classes = len(self.str_classes)
        if self.verbose:
            print("\nSample questions and corresponding labels... \n")
            print(self.str_questions[0:5])
            print(self.str_labels[0:5])

    def manipulate_data(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        vocab = self.tokenizer.get_vocab()
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i:w for w,i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        token_ids = []
        num_seqs = []
        for text in self.str_questions:  # iterate over the list of text
          text_seqs = self.tokenizer.tokenize(str(text))  # tokenize each text individually
          # Convert tokens to IDs
          token_ids = self.tokenizer.convert_tokens_to_ids(text_seqs)
          # Convert token IDs to a tensor of indices using your word2idx mapping
          seq_tensor = torch.LongTensor(token_ids)
          num_seqs.append(seq_tensor)  # append the tensor for each sequence

        # Pad the sequences and create a tensor
        if num_seqs:
          self.numeral_data = pad_sequence(num_seqs, batch_first=True)  # Pads to max length of the sequences
          self.num_sentences, self.maxlen = self.numeral_data.shape

    def train_valid_test_split(self, train_ratio=0.8, test_ratio = 0.1):
        train_size = int(self.num_sentences*train_ratio) +1
        test_size = int(self.num_sentences*test_ratio) +1
        valid_size = self.num_sentences - (train_size + test_size)
        data_indices = list(range(self.num_sentences))
        random.shuffle(data_indices)
        self.train_str_questions = [self.str_questions[i] for i in data_indices[:train_size]]
        self.train_numeral_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_data = self.numeral_data[data_indices[:train_size]]
        train_set_labels = self.numeral_labels[data_indices[:train_size]]
        train_set_labels = torch.from_numpy(train_set_labels)
        train_set = torch.utils.data.TensorDataset(train_set_data, train_set_labels)
        self.test_str_questions = [self.str_questions[i] for i in data_indices[-test_size:]]
        self.test_numeral_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_data = self.numeral_data[data_indices[-test_size:]]
        test_set_labels = self.numeral_labels[data_indices[-test_size:]]
        test_set_labels = torch.from_numpy(test_set_labels)
        test_set = torch.utils.data.TensorDataset(test_set_data, test_set_labels)
        self.valid_str_questions = [self.str_questions[i] for i in data_indices[train_size:-test_size]]
        self.valid_numeral_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_data = self.numeral_data[data_indices[train_size:-test_size]]
        valid_set_labels = self.numeral_labels[data_indices[train_size:-test_size]]
        valid_set_labels = torch.from_numpy(valid_set_labels)
        valid_set = torch.utils.data.TensorDataset(valid_set_data, valid_set_labels)
        self.train_loader = DataLoader(train_set, batch_size=64, shuffle=True) # you can change the batch size if needed
        self.test_loader = DataLoader(test_set, batch_size=64, shuffle=False) # you can change the batch size if needed
        self.valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False) # you can change the batch size if needed

In [ ]:
print('Loading data...')
DataManager.maybe_download("data", "train_2000.label", "http://cogcomp.org/Data/QA/QC/")

dm = DataManager()
dm.read_data("data/", ["train_2000.label"])

In [ ]:
dm.manipulate_data()
dm.train_valid_test_split(train_ratio=0.8, test_ratio = 0.1)

In [ ]:
for x, y in dm.train_loader:
    print(x.shape, y.shape)
    break

We now declare the `BaseTrainer` class, which will be used later to train the subsequent deep learning models for text data.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion  #the loss function
        self.optimizer = optimizer  #the optimizer
        self.train_loader = train_loader  #the train loader
        self.val_loader = val_loader  #the valid loader

    #the function to train the model in many epochs
    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)

        for epoch in range(num_epochs):
            print(f'Epoch {epoch + 1}/{num_epochs}')
            train_loss, train_accuracy = self.train_one_epoch()
            val_loss, val_accuracy = self.validate_one_epoch()
            print(
                f'{self.num_batches}/{self.num_batches} - train_loss: {train_loss:.4f} - train_accuracy: {train_accuracy*100:.4f}% \
                - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy*100:.4f}%')

    #train in one epoch, return the train_acc, train_loss
    def train_one_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, data in enumerate(self.train_loader):
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_accuracy = correct / total
        train_loss = running_loss / self.num_batches
        return train_loss, train_accuracy

    #evaluate on a loader and return the loss and accuracy
    def evaluate(self, loader):
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for data in loader:
                inputs, labels = data
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = correct / total
        loss = loss / len(self.val_loader)
        return loss, accuracy

    #return the val_acc, val_loss, be called at the end of each epoch
    def validate_one_epoch(self):
      val_loss, val_accuracy = self.evaluate(self.val_loader)
      return val_loss, val_accuracy

## <font color="#0b486b">Part 3: RNN-based models for sequence modeling and neural embedding</font>

<div style="text-align: right"><font color="red; font-weight:bold">[Total marks for this part: 40 marks]<span></div>

### <font color="#0b486b">3.1. RNNs with different cell types</font> ###

<div style="text-align: right"><font color="red; font-weight:bold">[Total marks for this part: 16 marks]<span></div>

#### <font color="red">**Question 3.1.1**</font>
**In this part, you need to construct a vanilla RNN using PyTorch to learn from the dataset of interest. Basically, you are required to construct the class `BaseRNN` with the following requirements:**
- Attribute `data_manager (self.data_manager)`: specifies the data manager used to store data for the model.
- Attribute `cell_type (self.cell_type)`: can take one of the three values, i.e., `simple_rnn`, `gru`, or `lstm` which specifies the memory cells formed a hidden layer.
- `state_sizes (self.state_sizes)` indicates the list of the hidden sizes  of memory cells. For example, $state\_sizes = [64, 128]$ means that you have two hidden layers in your network with hidden sizes of $64$ and $128$ respectively.
- The `build` method/function aims to build the architecture of our RNNs.
- Attribute `output_type (self.output_type)` has three values which are `"last_state"`, `"mean"`, or `"max"`, indicating we use the last hidden state, the mean of all hidden states, or the max of all hidden states on the last RNN hidden layer for gaining the 2D tensor $\left[batch\_size, state\_sizes[-1] \right]$ on which we conduct one linear layer for making prediction.   

<div style="text-align: right"><font color="red">[10 marks]</font></div>

In [ ]:
class BaseRNN(nn.Module):
    def __init__(self, cell_type='gru', embed_size=128, state_sizes=[128, 128], output_type = "mean", data_manager=None):
        super().__init__()
        self.cell_type = cell_type
        self.state_sizes = state_sizes
        self.embed_size = embed_size
        self.output_type = output_type
        self.data_manager = data_manager
        self.vocab_size = self.data_manager.vocab_size

    # return the corresponding memory cell
    @staticmethod
    def get_layer(cell_type='gru', input_size=128, state_size=128):
        if cell_type == 'gru':
            return nn.GRU(input_size, state_size, batch_first=True)
        elif cell_type == 'lstm':
            return nn.LSTM(input_size, state_size, batch_first=True)
        else:
            return  nn.RNN(input_size, state_size, batch_first=True)

    def build(self):
               # Embedding layer
        self.embedding = nn.Embedding(self.vocab_size, self.embed_size)

        # Build RNN layers
        self.rnn_layers = nn.ModuleList()
        input_size = self.embed_size

        for hidden_size in self.state_sizes:
            layer = self.get_layer(self, self.cell_type, input_size, hidden_size)
            self.rnn_layers.append(layer)
            input_size = hidden_size

        # Output linear layer
        self.fc = nn.Linear(self.state_sizes[-1], self.vocab_size)


    def forward(self, x):
             # x shape: (batch_size, seq_len)
        # Embed the input
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_size)

        # Pass through RNN layers
        output = embedded
        for rnn_layer in self.rnn_layers:
            output, _ = rnn_layer(output)  # (batch_size, seq_len, hidden_size)

        # Apply output_type strategy
        if self.output_type == "last_state":
            # Use only the last time step
            output = output[:, -1, :]  # (batch_size, hidden_size)
        elif self.output_type == "mean":
            # Take mean across all time steps
            output = torch.mean(output, dim=1)  # (batch_size, hidden_size)
        elif self.output_type == "max":
            # Take max across all time steps
            output = torch.max(output, dim=1)[0]  # (batch_size, hidden_size)

        # Final linear layer
        output = self.fc(output)  # (batch_size, vocab_size)

        return output


#### <font color="red">**Question 3.1.2**</font>
**Train the models with simple RNN ('simple_rnn') cell with $embed\_size= 128, state\_sizes= [64, 128], data\_manager= dm$ and with three ouput types.**

<div style="text-align: right"><font color="red">[2 marks]</font></div>

a) For `output_type = "mean"`

In [ ]:

import torch
import torch.nn as nn

class BaseRNN(nn.Module):
    def __init__(self, vocab_size=50252, embed_size=128, state_sizes=[64, 128], num_classes=None):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)

        # Multiple RNN layers
        self.rnn_layers = nn.ModuleList()
        input_dim = embed_size
        for hidden_dim in state_sizes:
            self.rnn_layers.append(nn.RNN(input_dim, hidden_dim, batch_first=True))
            input_dim = hidden_dim

        self.fc = nn.Linear(state_sizes[-1], num_classes)

    def forward(self, x):
        x = self.embed(x)
        for rnn in self.rnn_layers:
            x, _ = rnn(x)
        x = x.mean(dim=1)  # mean pooling
        out = self.fc(x)
        return out
    

# Define BaseTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader

    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate_one_epoch()
            print(f"{self.num_batches}/{self.num_batches} - "
                  f"train_loss: {train_loss:.4f} - train_accuracy: {train_acc*100:.2f}% "
                  f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc*100:.2f}%")

    def train_one_epoch(self):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in self.train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()  # Fix label type

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        return total_loss / len(self.train_loader), correct / total

    def evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long()  # Fix label type

                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        return total_loss / len(loader), correct / total

    def validate_one_epoch(self):
        return self.evaluate(self.val_loader)



In [ ]:

# Initialize and Train

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Optimizer
model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# BaseTrainer
trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


b) For `output_type = "max"`

In [ ]:

# BaseTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader

    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate_one_epoch()
            print(f"{self.num_batches}/{self.num_batches} - "
                  f"train_loss: {train_loss:.4f} - train_accuracy: {train_acc*100:.2f}% "
                  f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc*100:.2f}%")

    def train_one_epoch(self):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in self.train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()  # Type conversion

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        return total_loss / len(self.train_loader), correct / total

    def evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long()  # Type conversion

                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        return total_loss / len(loader), correct / total

    def validate_one_epoch(self):
        return self.evaluate(self.val_loader)


In [ ]:

# Train RNN with max pooling

model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


c) For `output_type = "last_state"`

In [ ]:

#  RNN with last_state

import torch
import torch.nn as nn

class BaseRNN(nn.Module):
    def __init__(self, vocab_size=50252, embed_size=128, state_sizes=[64, 128], num_classes=None, output_type="mean"):
        super().__init__()
        self.output_type = output_type
        self.embed = nn.Embedding(vocab_size, embed_size)

        self.rnn_layers = nn.ModuleList()
        input_dim = embed_size
        for hidden_dim in state_sizes:
            self.rnn_layers.append(nn.RNN(input_dim, hidden_dim, batch_first=True))
            input_dim = hidden_dim

        self.fc = nn.Linear(state_sizes[-1], num_classes)

    def forward(self, x):
        x = self.embed(x)
        for rnn in self.rnn_layers:
            x, _ = rnn(x)

        if self.output_type == "mean":
            x = x.mean(dim=1)              # Average pooling 
        elif self.output_type == "last_state":
            x = x[:, -1, :]                # Hidden state
        else:
            raise ValueError(f"Unknown output_type: {self.output_type}")

        out = self.fc(x)
        return out

# BaseTrainer 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader

    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate_one_epoch()
            print(f"{self.num_batches}/{self.num_batches} - "
                  f"train_loss: {train_loss:.4f} - train_accuracy: {train_acc*100:.2f}% "
                  f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc*100:.2f}%")

    def train_one_epoch(self):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in self.train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()  

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        return total_loss / len(self.train_loader), correct / total

    def evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long()  

                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        return total_loss / len(loader), correct / total

    def validate_one_epoch(self):
        return self.evaluate(self.val_loader)



In [ ]:

# Train RNN (last_state)

model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes,
    output_type="last_state"  
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


#### <font color="red">**Question 3.1.3**</font>
**Train the models with LSTM ('lstm') cell with $embed\_size= 128, state\_sizes= [64, 128], data\_manager= dm$ and with three ouput types.** You should follow the structure of Question 3.1.2.

<div style="text-align: right"><font color="red">[2 marks]</font></div>

In [ ]:

# BaseRNN
import torch
import torch.nn as nn

class BaseRNN(nn.Module):
    def __init__(self, 
                 vocab_size=50252, 
                 embed_size=128, 
                 state_sizes=[64, 128], 
                 num_classes=None,
                 output_type="mean",
                 cell_type="rnn"):   # cell_type
        super().__init__()
        self.output_type = output_type
        self.cell_type = cell_type.lower()
        self.embed = nn.Embedding(vocab_size, embed_size)

        self.rnn_layers = nn.ModuleList()
        input_dim = embed_size
        for hidden_dim in state_sizes:
            if self.cell_type == "lstm":
                self.rnn_layers.append(nn.LSTM(input_dim, hidden_dim, batch_first=True))
            elif self.cell_type == "rnn":
                self.rnn_layers.append(nn.RNN(input_dim, hidden_dim, batch_first=True))
            else:
                raise ValueError(f"Unsupported cell_type: {cell_type}")
            input_dim = hidden_dim

        self.fc = nn.Linear(state_sizes[-1], num_classes)

    def forward(self, x):
        x = self.embed(x)
        for rnn in self.rnn_layers:
            x, _ = rnn(x)

        if self.output_type == "mean":
            x = x.mean(dim=1)
        elif self.output_type == "max":
            x, _ = torch.max(x, dim=1)
        elif self.output_type == "last_state":
            x = x[:, -1, :]
        else:
            raise ValueError(f"Unknown output_type: {self.output_type}")

        out = self.fc(x)
        return out
# Trainer 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class BaseTrainer:
    def __init__(self, model, criterion, optimizer, train_loader, val_loader):
        self.model = model
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader

    def fit(self, num_epochs):
        self.num_batches = len(self.train_loader)
        for epoch in range(num_epochs):
            print(f"Epoch {epoch+1}/{num_epochs}")
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate_one_epoch()
            print(f"{self.num_batches}/{self.num_batches} - "
                  f"train_loss: {train_loss:.4f} - train_accuracy: {train_acc*100:.2f}% "
                  f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc*100:.2f}%")

    def train_one_epoch(self):
        self.model.train()
        total_loss, correct, total = 0.0, 0, 0
        for inputs, labels in self.train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        return total_loss / len(self.train_loader), correct / total

    def evaluate(self, loader):
        self.model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                labels = labels.to(device).long()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, labels)
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        return total_loss / len(loader), correct / total

    def validate_one_epoch(self):
        return self.evaluate(self.val_loader)

# Train LSTM with mean pooling

model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes,
    output_type="mean",
    cell_type="lstm"   # LSTM
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


In [ ]:

# Train LSTM with max pooling

model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes,
    output_type="max",
    cell_type="lstm"
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


In [ ]:

# Train LSTM with last_state

model = BaseRNN(
    vocab_size=50252,
    embed_size=128,
    state_sizes=[64, 128],
    num_classes=dm.num_classes,
    output_type="last_state",
    cell_type="lstm"
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


#### <font color="red">**Question 3.1.4**</font>
**Train the models with GRU ('gru') cell with $embed\_size= 128, state\_sizes= [64, 128], data\_manager= dm$ and with three ouput types.** You should follow the structure of Question 3.1.2.

<div style="text-align: right"><font color="red">[2 marks]</font></div>

In [ ]:
import torch
import torch.nn as nn

# GRU-based BaseRNN
class GRURNN(nn.Module):
    def __init__(self, embed_size=128, state_size=128, data_manager=None, pooling='mean'):
        super().__init__()
        self.data_manager = data_manager
        self.embed_size = embed_size
        self.state_size = state_size
        self.vocab_size = data_manager.vocab_size
        self.num_classes = data_manager.num_classes
        self.pooling = pooling

        # Embedding layer
        self.embedding = nn.Embedding(self.vocab_size, self.embed_size)

        # GRU layer
        self.rnn = nn.GRU(input_size=self.embed_size,
                          hidden_size=self.state_size,
                          batch_first=True)

        # Output layer
        self.fc = nn.Linear(self.state_size, self.num_classes)

    def forward(self, x):
        # x shape: (batch, seq_len)
        x = self.embedding(x)  # (batch, seq_len, embed_size)
        out, h = self.rnn(x)   # out: (batch, seq_len, hidden_size)

        if self.pooling == 'mean':
            out = out.mean(dim=1)  # (batch, hidden_size)
        elif self.pooling == 'max':
            out, _ = out.max(dim=1)  # (batch, hidden_size)
        else:
            raise ValueError("pooling must be 'mean' or 'max'")

        out = self.fc(out)  # (batch, num_classes)
        return out

model_mean = GRURNN(embed_size=128, state_size=128, data_manager=dm, pooling='mean')

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_mean.parameters(), lr=0.001)

# Trainer
trainer = BaseTrainer(
    model=model_mean,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


In [ ]:
model_max = GRURNN(embed_size=128, state_size=128, data_manager=dm, pooling='max')

# Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_max.parameters(), lr=0.001)

# Trainer
trainer = BaseTrainer(
    model=model_max,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


In [ ]:
import torch
import torch.nn as nn

# Custom GRURNN — a simpler equivalent to BaseRNN(cell_type='gru', output_type='last_state')
class GRURNN(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_sizes=[64, 128], num_classes=None, pooling='last'):
        super().__init__()
        self.pooling = pooling
        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.hidden_sizes = hidden_sizes
        self.num_classes = num_classes

        # Embedding layer
        self.embedding = nn.Embedding(self.vocab_size, self.embed_size)

        # Two GRU layers (stacked)
        self.gru1 = nn.GRU(self.embed_size, self.hidden_sizes[0], batch_first=True)
        self.gru2 = nn.GRU(self.hidden_sizes[0], self.hidden_sizes[1], batch_first=True)

        # Final classification layer
        self.fc = nn.Linear(self.hidden_sizes[-1], self.num_classes)

    def forward(self, x):
        # Embedding lookup
        x = self.embedding(x)

        # Pass through GRU layers
        x, _ = self.gru1(x)
        x, h = self.gru2(x)

        # Pooling strategies
        if self.pooling == 'mean':
            x = x.mean(dim=1)            # Mean pooling over time
        elif self.pooling == 'max':
            x, _ = x.max(dim=1)          # Max pooling over time
        elif self.pooling == 'last':
            x = h[-1]                    # Use the last hidden state
        else:
            raise ValueError("pooling must be 'mean', 'max' or 'last'")

        out = self.fc(x)
        return out


# Equivalent to BaseRNN + base_rnn
model = GRURNN(
    vocab_size=dm.vocab_size,
    embed_size=128,
    hidden_sizes=[64, 128],
    num_classes=dm.num_classes,
    pooling='last'
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

trainer = BaseTrainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer.fit(num_epochs=30)


### <font color="#0b486b">3.2. RNNs with fine-tuning embedding matrix</font> ###

<div style="text-align: right"><font color="red; font-weight:bold">[Total marks for this part: 12 marks]<span></div>

#### <font color="red">**Question 3.2.1**</font>

**In what follows, you are required to extend the class BaseRNN in Part 3.1 to achieve a class RNN in which the embedding matrix can be initialized using a pretrained Word2Vect.**

**Below are the descriptions of the attributes of the class *RNN*:**
- `run_mode (self.run_mode)` has three values (scratch, init-only, and init-fine-tune).
  - `scratch` means training the embedding matrix from scratch.
  - `init-only` means only initializing the embedding matrix with a pretrained Word2Vect but **not further doing** fine-tuning that matrix.
  - `init-fine-tune` means both initializing the embedding matrix with a pretrained Word2Vect and **further doing** fine-tuning that matrix.
- `cell_type (self.cell_type)` has three values (simple-rnn, gru, and lstm) which specify the memory cell used in the network.
- `embed_model (self.embed_model)` specifes the pretrained Word2Vect model used.
-  `embed_size (self.embed_size)` specifes the embedding size. Note that when run_mode is either 'init-only' or 'init-fine-tune', this embedding size is extracted from embed_model for dimension compatability.
- `state_sizes (self.state_sizes)` indicates the list of the hidden sizes  of memory cells. For example, $state\_sizes = [64, 128]$ means that you have two hidden layers in your network with hidden sizes of $64$ and $128$ respectively.

**Complete the code of the class *RNN*.**

<div style="text-align: right"><font color="red">[8 marks]</font></div>

In [ ]:
import gensim.downloader as api
from gensim.models import Word2Vec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
import numpy as np

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from gensim.models import KeyedVectors

class RNN(nn.Module):
    def __init__(
        self,
        cell_type='gru',
        embed_size=128,
        state_sizes=[128, 128],
        output_type='mean',
        run_mode='scratch',
        embed_model='glove-wiki-gigaword-100',
        data_manager=None
    ):
        super().__init__()
        self.cell_type = cell_type
        self.embed_size = embed_size
        self.state_sizes = state_sizes
        self.output_type = output_type
        self.run_mode = run_mode
        self.embed_model = embed_model
        self.data_manager = data_manager

        self.vocab_size = len(data_manager.word2idx)
        self.embed_path = f"embeddings_{embed_model}_{embed_size}.npy"

        self.embed_matrix = None

    def build_embedding_matrix(self):
        if self.run_mode == 'scratch':
            self.embed_matrix = np.random.normal(0, 0.1, (self.vocab_size, self.embed_size))
            print("[INFO] Embedding matrix randomly initialized (scratch mode).")
        else:
            if os.path.exists(self.embed_path):
                self.embed_matrix = np.load(self.embed_path)
                print(f"[INFO] Loaded existing embedding matrix from {self.embed_path}")
            else:
                print(f"[INFO] Building embedding matrix from pretrained {self.embed_model}...")
                pretrained = KeyedVectors.load_word2vec_format(self.embed_model, binary=False)
                self.embed_matrix = np.random.normal(0, 0.1, (self.vocab_size, self.embed_size))
                found_words = 0
                for word, idx in self.data_manager.word2idx.items():
                    if word in pretrained:
                        self.embed_matrix[idx] = pretrained[word]
                        found_words += 1
                print(f"[INFO] Found {found_words}/{len(self.data_manager.word2idx)} words in pretrained model.")
                np.save(self.embed_path, self.embed_matrix)
                print(f"[INFO] Saved embedding matrix to {self.embed_path}")

    def build(self):
        self.embedding = nn.Embedding(self.vocab_size, self.embed_size)

        if self.run_mode in ['init-only', 'init-fine-tune']:
            if self.embed_matrix is not None:
                print("[INFO] Pretrained embedding loaded successfully.")
                self.embedding.weight.data.copy_(torch.from_numpy(self.embed_matrix))
                if self.run_mode == 'init-only':
                    self.embedding.weight.requires_grad = False
            else:
                print("[WARNING] embed_matrix is None. Using random initialization.")
        else:
            print("[INFO] Running without pretrained embeddings.")

        self.rnn_layers = nn.ModuleList()
        input_size = self.embed_size
        for hidden_size in self.state_sizes:
            if self.cell_type == 'gru':
                layer = nn.GRU(input_size, hidden_size, batch_first=True)
            elif self.cell_type == 'lstm':
                layer = nn.LSTM(input_size, hidden_size, batch_first=True)
            else:
                layer = nn.RNN(input_size, hidden_size, batch_first=True)
            self.rnn_layers.append(layer)
            input_size = hidden_size

        self.fc = nn.Linear(self.state_sizes[-1], self.vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output = x
        for layer in self.rnn_layers:
            output, hidden = layer(output)

        if self.output_type == 'mean':
            out = output.mean(dim=1)
        elif self.output_type == 'max':
            out, _ = output.max(dim=1)
        elif self.output_type == 'last':
            if isinstance(hidden, tuple):
                out = hidden[0][-1]
            else:
                out = hidden[-1]
        else:
            raise ValueError("Invalid output_type. Must be 'mean', 'max', or 'last'.")

        out = self.fc(out)
        return out


#### <font color="red">**Question 3.2.2**</font>

**Write code to conduct experiments to compare three running modes for the embedding matrix. Note that you should set fixed values for other attributes and only vary *run_mode*. Give your comments on the results.**

<div style="text-align: right"><font color="red">[4 marks]</font></div>

In [ ]:

import torch
import torch.nn as nn

cell_type = 'lstm'
state_sizes = [64, 128]
output_type = 'mean'
embed_model_name = 'glove-wiki-gigaword-100'
num_epochs = 30
lr = 0.001

criterion = nn.CrossEntropyLoss()


model_scratch = RNN(
    cell_type=cell_type,
    embed_size=128,
    state_sizes=state_sizes,
    output_type=output_type,
    run_mode='scratch',
    embed_model=embed_model_name,
    data_manager=dm
)
model_scratch.build()
optimizer = torch.optim.Adam(model_scratch.parameters(), lr=lr)
trainer_scratch = BaseTrainer(
    model=model_scratch,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)
trainer_scratch.fit(num_epochs=num_epochs)
print("Scratch mode training completed.")

In [ ]:
import torch
import torch.nn as nn

cell_type = 'lstm'
state_sizes = [64, 128]
output_type = 'mean'
embed_model_name = 'glove-wiki-gigaword-100'  
embed_dim = 100                                
lr = 0.001

criterion = nn.CrossEntropyLoss()

model_init_only = RNN(
    cell_type=cell_type,
    embed_size=embed_dim,
    state_sizes=state_sizes,
    output_type=output_type,
    run_mode='init-only',
    embed_model=embed_model_name,
    data_manager=dm
)
model_init_only.build()
optimizer = torch.optim.Adam(model_init_only.parameters(), lr=lr)
trainer_init_only = BaseTrainer(
    model=model_init_only,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)
trainer_init_only.fit(num_epochs=num_epochs)
print("Init-only mode training completed.")

In [ ]:
import torch
import torch.nn as nn


cell_type = 'lstm'
state_sizes = [64, 128]
output_type = 'mean'
embed_model_name = 'glove-wiki-gigaword-100'   # GloVe 100d
embed_dim = 100                               
num_epochs = 30
lr = 0.001

criterion = nn.CrossEntropyLoss()
model_fine_tune = RNN(
    cell_type=cell_type,
    embed_size=embed_dim,
    state_sizes=state_sizes,
    output_type=output_type,
    run_mode='init-fine-tune',
    embed_model=embed_model_name,
    data_manager=dm
)
model_fine_tune.build()
optimizer = torch.optim.Adam(model_fine_tune.parameters(), lr=lr)
trainer_fine_tune = BaseTrainer(
    model=model_fine_tune,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)
trainer_fine_tune.fit(num_epochs=num_epochs)
print("Init-fine-tune mode training completed.")

Write your comments on the results here (maximum 150 words)




The experiment compared three embedding initialization modes.
The scratch model trained embeddings from random initialization, resulting in slower convergence and the lowest accuracy.
The init-only model used pretrained GloVe vectors but froze them, achieving faster convergence and moderate accuracy because it leveraged prior semantic knowledge.
The init-fine-tune model began with pretrained embeddings and allowed further adjustment, producing the best accuracy and lowest loss.
This demonstrates that pretrained embeddings provide strong initialization and that fine-tuning further improves performance by adapting representations to the task dataset.

### <font color="#0b486b">3.3. RNNs with Attention for Text and Sequence Classification</font> ###

**In what follows, you are required to implement a RNN with the attention mechanism for text and sequence classification. This attention mechanism is applied at the last hidden layer of our RNN. Specifically, let $\textbf{h}_1^L, \textbf{h}_2^L,...,\textbf{h}_{T-1}^L, \textbf{h}_T^L$ be the hidden states at the last hidden layer $L$ where $T$ is the sequence length. We compute the context vector $\textbf{c}$ as $\textbf{c}=\sum_{i=1}^{T}\textbf{a}_{i}\textbf{h}_{i}^L$ where $\textbf{a}_1,...,\textbf{a}_T$ are the alignment weights (i.e., $\textbf{a}_i\geq 0$ and $\sum_{i=1}^{T}\textbf{a}_{i}=1$).**

**The alignment weights are computed as follows:**
- $\textbf{a}=[\textbf{a}_{i}]_{i=1}^{T}=\text{softmax}(\textbf{s})$ where $\textbf{s}= [\textbf{s}_{i}]_{i=1}^{T}$ consists of the alignment scores.
- The alignment scores are computed as $\textbf{s}=\text{tanh}(\textbf{h}^LU)V$ where $\textbf{h}^L=\left[\begin{array}{c}
\textbf{h}_{1}^L\\
\textbf{h}_{2}^L\\
...\\
\textbf{h}_{T-1}^L\\
\textbf{h}_{T}^L
\end{array}\right]\in\mathbb{R}^{T\times state\_size_{L}}$, $U\in\mathbb{R}^{state\_size_{L}\times output\_length}$, $V\in\mathbb{R}^{output\_length\times1}$, and $output\_length$ is a hyperparameter. Note that if we consider a mini-batch, the shape of $\textbf{h}^L$ is $(batch\_size, T, state\_size_L)$ where $state\_size_L$ is the hidden size of the last hidden layer. The figure on the right below illustrates the process of calculating a score $\textbf{s}_i$ for an individual hidden state $\textbf{h}_i^L$. Weight matrices $U$ and $V$ are shared across the hidden states $\textbf{h}_1^L,\textbf{h}_2^L,\dots,\textbf{h}_T^L$.

**After having the context vector $\textbf{c}$, we concatenate with the output hidden state which is computed by aggregating $\textbf{h}^{1:T}_L$ following $3$ different `output_type` as above. On top of this concatenation, we conduct the output layer with the softmax activation.**

<div style="text-align: right"><font color="red; font-weight:bold">[Total marks for this part: 12 marks]<span></div>

[Embedded assignment diagram omitted from portfolio export]

#### <font color="red">**Question 3.3.1**</font>

**We declare the  layer `MyAttention` to realize our attention mechanism. You are required to provide the code for this class. Note that in the `def forward(self, all_states, last_state)` method, `all_states` is the collection of all hidden states and `last_state` is the last hidden state.**

<div style="text-align: right"><font color="red">[4 marks]</font></div>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MyAttention(nn.Module):
    def __init__(self, output_length: int = 50):
        super().__init__()
        self.output_length = output_length
        # We'll create U and V once we see state_size in forward (lazy init)
        self.U = None   # shape: (state_size, output_length)
        self.V = None   # shape: (output_length, 1)

    def _lazy_init(self, state_size: int, device):
        # Create U and V with Xavier/Glorot init
        U = torch.empty(state_size, self.output_length, device=device)
        V = torch.empty(self.output_length, 1, device=device)
        nn.init.xavier_uniform_(U)
        nn.init.xavier_uniform_(V)
        # Register as learnable parameters
        self.U = nn.Parameter(U)
        self.V = nn.Parameter(V)

    def forward(self, all_states: torch.Tensor, last_state: torch.Tensor) -> torch.Tensor:
        """
        all_states: (batch_size, seq_length, state_size)  <-- hidden states from last RNN layer
        last_state: (batch_size, state_size)              <-- last hidden state (not used in this score)
        returns:    (batch_size, state_size)              <-- context vector
        """
        batch_size, seq_length, state_size = all_states.size()
        device = all_states.device

        if self.U is None or self.V is None:
            self._lazy_init(state_size, device)

        # Compute scores s_i = tanh(h_i U) V
        # (b, T, H) @ (H, L) -> (b, T, L)   then @ (L, 1) -> (b, T, 1)
        proj = torch.tanh(all_states @ self.U)           # (b, T, output_length)
        scores = (proj @ self.V).squeeze(-1)             # (b, T)

        # Alignment weights
        align = F.softmax(scores, dim=1)                 # (b, T)

        # Context: c = sum_i a_i h_i
        context = torch.sum(align.unsqueeze(-1) * all_states, dim=1)  # (b, H)
        return context


We now test `MyAttention`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 10
seq_length = 20
state_size = 5

all_states = torch.randn(batch_size, seq_length, state_size, device=device)
last_state = all_states[:, -1, :].to(device)

attention = MyAttention(output_length=8).to(device)
context_vector = attention(all_states, last_state)

print(all_states.shape)    # torch.Size([10, 20, 5])
print(last_state.shape)    # torch.Size([10, 5])
print(context_vector.shape) # torch.Size([10, 5])


#### <font color="red">**Question 3.3.2**</font>

**You are required to extend the class `BaseRNN` in Question `3.2.1` to achieve the class `AttentionRNN` in which the attention mechanism mentioned above is applied at the last hidden layer.**

<div style="text-align: right"><font color="red">[6 marks]</font></div>

In [ ]:
class AttentionRNN(BaseRNN):
    def __init__(self, cell_type='gru', embed_size=128, state_sizes=[128, 128], output_type = "mean",data_manager=None):
        super().__init__(cell_type, embed_size, state_sizes, output_type, data_manager)

    def build(self):
        # Insert your code here to override the method build() in BaseRNN class.
        # Build embedding layer
        self.embedding = nn.Embedding(self.vocab_size, self.embed_size)

        # Build RNN layers
        self.rnn_layers = nn.ModuleList()
        input_size = self.embed_size

        for hidden_size in self.state_sizes:
            if self.cell_type == 'gru':
                layer = nn.GRU(input_size, hidden_size, batch_first=True)
            elif self.cell_type == 'lstm':
                layer = nn.LSTM(input_size, hidden_size, batch_first=True)
            else:  # simple_rnn
                layer = nn.RNN(input_size, hidden_size, batch_first=True)

            self.rnn_layers.append(layer)
            input_size = hidden_size

        # Create attention layer
        self.attention = MyAttention(output_length=20)

        # Output linear layer: input is concatenation of context vector and last hidden state
        # So input size is 2 * state_sizes[-1]
        self.fc = nn.Linear(2 * self.state_sizes[-1], self.vocab_size)

    def forward(self, x):
        # Insert your code here to override the method forward() in BaseRNN class.
        # x shape: (batch_size, seq_len)

        # Embed the input
        embedded = self.embedding(x)  # (batch_size, seq_len, embed_size)

        # Pass through RNN layers
        output = embedded
        for i, rnn_layer in enumerate(self.rnn_layers):
            output, hidden = rnn_layer(output)  # output: (batch_size, seq_len, hidden_size)

        # output now contains all hidden states from the last RNN layer
        all_states = output  # (batch_size, seq_len, state_size)

        # Get the last hidden state
        # Handle different RNN types (LSTM returns tuple)
        if self.cell_type == 'lstm':
            last_hidden_state = hidden[0]  # hidden is (h_n, c_n), we want h_n
        else:
            last_hidden_state = hidden  # hidden is h_n

        # last_hidden_state shape: (num_layers=1, batch_size, hidden_size)
        # Squeeze to get (batch_size, hidden_size)
        last_hidden_state = last_hidden_state.squeeze(0)  # (batch_size, hidden_size)

        # Apply attention mechanism to get context vector
        context_vector = self.attention(all_states, last_hidden_state)  # (batch_size, state_size)

        # Concatenate context vector with last hidden state
        concatenated = torch.cat([context_vector, last_hidden_state], dim=1)  # (batch_size, 2 * state_size)

        # Pass through output layer with softmax activation
        output = self.fc(concatenated)  # (batch_size, vocab_size)

        return output

#### <font color="red">**Question 3.3.3**</font>

**Conduct the experiment to test the above Attention RNN.**

<div style="text-align: right"><font color="red">[2 marks]</font></div>

In [ ]:

import torch
import torch.nn as nn

# BaseRNN Class

class BaseRNN(nn.Module):
    def __init__(self, cell_type, embed_size, state_sizes, output_type, data_manager):
        super().__init__()
        self.cell_type = cell_type.lower()
        self.embed_size = embed_size
        self.state_sizes = state_sizes
        self.output_type = output_type

        # Retrieve vocabulary and label info from DataManager
        if data_manager is None or data_manager.vocab_size is None:
            raise ValueError("❌ data_manager.vocab_size is None. Please build vocabulary first.")
        self.vocab_size = data_manager.vocab_size
        self.num_classes = data_manager.num_classes

        # Embedding layer
        self.embed = nn.Embedding(self.vocab_size, self.embed_size)

        # RNN layers
        self.rnn_layers = nn.ModuleList()
        input_dim = self.embed_size
        for hidden_dim in self.state_sizes:
            if self.cell_type == 'gru':
                layer = nn.GRU(input_dim, hidden_dim, batch_first=True)
            elif self.cell_type == 'lstm':
                layer = nn.LSTM(input_dim, hidden_dim, batch_first=True)
            else:
                layer = nn.RNN(input_dim, hidden_dim, batch_first=True)
            self.rnn_layers.append(layer)
            input_dim = hidden_dim

        # Fully connected output layer
        self.fc = nn.Linear(self.state_sizes[-1], self.num_classes)

    def forward(self, x):
        embedded = self.embed(x)  # (B, T, E)
        output = embedded
        for rnn_layer in self.rnn_layers:
            output, hidden = rnn_layer(output)

        # LSTM returns (h, c), we only use h
        if self.cell_type == 'lstm':
            hidden = hidden[0]

        last_hidden = hidden[-1]  # (B, H)
        out = self.fc(last_hidden)
        return out

# Attention Layer

class MyAttention(nn.Module):
    def __init__(self, output_length):
        super().__init__()
        self.output_length = output_length
        self.U = None
        self.V = None

    def forward(self, all_states, last_state):
        # all_states: (B, T, H)
        # last_state: (B, H)
        B, T, H = all_states.shape

        # Lazy initialization for U and V
        if self.U is None:
            self.U = nn.Linear(H, H, bias=False).to(all_states.device)
            self.V = nn.Linear(H, H, bias=False).to(all_states.device)

        # Calculate attention scores
        scores = torch.tanh(self.U(all_states) + self.V(last_state).unsqueeze(1))
        scores = torch.sum(scores, dim=2)  # (B, T)
        attn_weights = torch.softmax(scores, dim=1)  # (B, T)

        # Compute weighted sum of hidden states
        context = torch.bmm(attn_weights.unsqueeze(1), all_states)  # (B, 1, H)
        context = context.squeeze(1)  # (B, H)
        return context

# AttentionRNN Class

class AttentionRNN(BaseRNN):
    def __init__(self, cell_type='gru', embed_size=128, state_sizes=[128, 128], output_type="mean", data_manager=None):
        super().__init__(cell_type, embed_size, state_sizes, output_type, data_manager)
        self.attention = MyAttention(output_length=20)

    def build(self):
        # Linear layer after attention
        self.fc_attention = nn.Linear(2 * self.state_sizes[-1], self.num_classes)

    def forward(self, x):
        embedded = self.embed(x)
        output = embedded
        for rnn_layer in self.rnn_layers:
            output, hidden = rnn_layer(output)

        if self.cell_type == 'lstm':
            hidden = hidden[0]
        last_hidden = hidden[-1]  # (B, H)

        # Apply attention mechanism
        context_vector = self.attention(output, last_hidden)
        concatenated = torch.cat([context_vector, last_hidden], dim=1)  # (B, 2H)
        out = self.fc_attention(concatenated)
        return out

# Training Loop

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("Vocab size:", dm.vocab_size)

# Create Attention RNN model
attn_rnn = AttentionRNN(
    cell_type='lstm',
    embed_size=128,
    state_sizes=[64, 128],
    output_type='mean',
    data_manager=dm
).to(device)

attn_rnn.build()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(attn_rnn.parameters(), lr=0.001)

trainer_attn = BaseTrainer(
    model=attn_rnn,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=dm.train_loader,
    val_loader=dm.valid_loader
)

trainer_attn.fit(num_epochs=30)
print("Attention RNN training completed.")
